In [ ]:
!pip install -U -q bitsandbytes accelerate transformers

In [ ]:
import torch
import json
import os
from transformers import AutoModelForCausalLM, AutoTokenizer, GenerationConfig, BitsAndBytesConfig
from sentence_transformers import SentenceTransformer, util
from tqdm import tqdm

# 1. Настройка моделей
MODEL_NAME = "IlyaGusev/saiga_llama3_8b"
NEUTRAL_SYSTEM_PROMPT = "Ты — лингвистический ассистент. Перескажи текст сухим, современным и информационным языком. Удали архаизмы и эмоции Достоевского. Оставь только факты."

# Загружаем оценщика (он маленький, поместится в память вместе с 8bit моделью)
embedder = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')

bnb_config = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_threshold=6.0,
    llm_int8_enable_fp32_cpu_offload=True
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16
)
model.eval()

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
generation_config = GenerationConfig.from_pretrained(MODEL_NAME)

# 2. Загрузка датасета (Путь из INPUT)
input_path = '/kaggle/input/datasets/kirillsilantev/dostoevsky-fragments/dostoevsky_original_fragments1.json'
with open(input_path, 'r', encoding='utf-8') as f:
    dataset = json.load(f)

# 3. Обработка фрагментов
print("Начинаем нейтрализацию...")
results = []
for item in tqdm(dataset[:10]): # Тестируем на 10 фрагментах
    original_text = item['original_fragment']
    
    prompt = tokenizer.apply_chat_template([
        {"role": "system", "content": NEUTRAL_SYSTEM_PROMPT},
        {"role": "user", "content": f"Текст: {original_text}"}
    ], tokenize=False, add_generation_prompt=True)
    
    data = tokenizer(prompt, return_tensors="pt", add_special_tokens=False).to(model.device)
    
    with torch.no_grad():
        output_ids = model.generate(**data, generation_config=generation_config)[0]
    
    output_ids = output_ids[len(data["input_ids"][0]):]
    neutral_output = tokenizer.decode(output_ids, skip_special_tokens=True).strip()
    
    # Считаем Topic Fidelity (Косинусное сходство)
    emb_orig = embedder.encode(original_text, convert_to_tensor=True)
    emb_neut = embedder.encode(neutral_output, convert_to_tensor=True)
    cos_sim = util.cos_sim(emb_orig, emb_neut).item()
    
    item['neutral_proxy'] = neutral_output
    item['topic_fidelity'] = round(cos_sim, 4)
    results.append(item)

# 4. Сохранение (В WORKING директорию!)
output_path = '/kaggle/working/dostoevsky_parallel_dataset.json'
with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(results, f, ensure_ascii=False, indent=4)

print(f"\nТест завершен. Средний Score: {sum(i['topic_fidelity'] for i in results)/len(results):.4f}")

In [ ]:
import json

#загружаем сохраненный результат
output_path = '/kaggle/working/dostoevsky_parallel_dataset.json'
with open(output_path, 'r', encoding='utf-8') as f:
    results = json.load(f)

for i, item in enumerate(results):
    print(f"фрагмент {i+1}, скор: {item['topic_fidelity']}")
    print("-" * 30)
    
    print("оригинал:")
    print(item['original_fragment'])
    
    print("\nнейтральный:")
    print(item['neutral_proxy'])
    print(f"{'-'*100}\n")

In [ ]:
import torch
import json
from tqdm import tqdm
from sentence_transformers import SentenceTransformer, util

# 1. Настройки батчинга
BATCH_SIZE = 8  # Можно попробовать 16, если хватит VRAM
INPUT_FILE = '/kaggle/input/datasets/kirillsilantev/dostoevsky-fragments/dostoevsky_original_fragments1.json'
OUTPUT_FILE = '/kaggle/working/dostoevsky_parallel_dataset_final.json'

# Настройка токенайзера для батчинга
tokenizer.padding_side = "left" 
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# 2. Подготовка данных
with open(INPUT_FILE, 'r', encoding='utf-8') as f:
    dataset = json.load(f)

final_data = []

# 3. Ускоренный цикл обработки
model.eval()
for i in tqdm(range(0, len(dataset), BATCH_SIZE), desc="Batch Processing"):
    batch = dataset[i : i + BATCH_SIZE]
    
    # Формируем промпты для всей пачки
    prompts = []
    for item in batch:
        messages = [
            {"role": "system", "content": NEUTRAL_SYSTEM_PROMPT},
            {"role": "user", "content": f"Текст для упрощения: {item['original_fragment']}"}
        ]
        prompts.append(tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True))
    
    # Токенизация всей пачки сразу
    inputs = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True, max_length=512).to(model.device)
    
    # Генерация для всей пачки
    with torch.no_grad():
        output_ids = model.generate(
            **inputs, 
            generation_config=generation_config,
            max_new_tokens=256,
            pad_token_id=tokenizer.pad_token_id
        )
    
    # Декодирование результатов
    for j, item in enumerate(batch):
        # Извлекаем только ответ ассистента, обрезая входной промпт
        generated_text = tokenizer.decode(output_ids[j][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
        item['neutral_proxy'] = generated_text
        final_data.append(item)

# 4. Массовая оценка Topic Fidelity (так быстрее, чем в цикле)
print("Начинаю массовую оценку сходства...")
originals = [item['original_fragment'] for item in final_data]
neutrals = [item['neutral_proxy'] for item in final_data]

# Эмбеддер сам эффективно использует GPU для батчей
emb_orig = embedder.encode(originals, batch_size=32, show_progress_bar=True, convert_to_tensor=True)
emb_neut = embedder.encode(neutrals, batch_size=32, show_progress_bar=True, convert_to_tensor=True)

# Считаем косинусное сходство попарно
cosine_scores = util.cos_sim(emb_orig, emb_neut).diagonal().tolist()

for i, score in enumerate(cosine_scores):
    final_data[i]['topic_fidelity'] = round(score, 4)
    final_data[i]['is_valid'] = score > 0.75

# 5. Сохранение
with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
    json.dump(final_data, f, ensure_ascii=False, indent=4)

print(f"Готово! Средний скор по всему датасету: {sum(cosine_scores)/len(cosine_scores):.4f}")

saiga не работает в параллельных потоках, для этого нужна LLama с vLLM, поэтому дальше работаю с LLama

In [ ]:
# Ставим конкретную стабильную версию
!pip uninstall -y vllm flashinfer
!pip install -q vllm==0.6.3 sentence-transformers accelerate

In [ ]:
import os
import json
import torch
from vllm import LLM, SamplingParams
from sentence_transformers import SentenceTransformer, util
from kaggle_secrets import UserSecretsClient

import os
# ФИКС PROTOBUF: заставляем использовать Python-реализацию, чтобы убрать ошибку GetPrototype
os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"
os.environ["VLLM_ATTENTION_BACKEND"] = "XFORMERS"

import torch
import json
# ... дальше твои импорты vllm и остальное

# Флаг, чтобы vLLM не искал flashinfer
os.environ["VLLM_ATTENTION_BACKEND"] = "XFORMERS"

# --- НАСТРОЙКИ ---
MODEL_NAME = "meta-llama/Meta-Llama-3-8B-Instruct"
SYSTEM_PROMPT = (
    "Ты — лингвистический робот. Перепиши текст современным нейтральным русским языком, "
    "сохранив ВСЕ факты. Удали авторский пафос и архаизмы Достоевского."
)

# --- ИНИЦИАЛИЗАЦИЯ ---
user_secrets = UserSecretsClient()
os.environ["HF_TOKEN"] = user_secrets.get_secret("HF_TOKEN")

# Настройка vLLM, которая НЕ требует компиляции при старте
llm = LLM(
    model=MODEL_NAME,
    tensor_parallel_size=2,      # 2x T4
    gpu_memory_utilization=0.85, 
    max_model_len=1024,
    enforce_eager=True,           # КРИТИЧНО: отключает JIT и поиск -lcuda
    disable_custom_all_reduce=True,
    dtype="float16"
)

embedder = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2', device="cuda:0")
sampling_params = SamplingParams(temperature=0.0, max_tokens=512)

In [ ]:
INPUT_PATH = '/kaggle/input/datasets/kirillsilantev/dostoevsky-fragments/dostoevsky_original_fragments1.json'
with open(INPUT_PATH, 'r', encoding='utf-8') as f:
    dataset = json.load(f)

def get_prompt(text):
    return f"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\n{SYSTEM_PROMPT}<|eot_id|>" \
           f"<|start_header_id|>user<|end_header_id|>\n\nТекст: {text}<|eot_id|>" \
           f"<|start_header_id|>assistant<|end_header_id|>\n\n"

In [ ]:
# На всякий случай определим переменные здесь, если они не подтянулись
TEST_LIMIT = 10

print(f"\n{'='*20} ЭТАП 1: ТЕСТОВЫЙ ЗАПУСК (10 примеров) {'='*20}")

# Берем первые 10 элементов
test_items = dataset[:TEST_LIMIT]

# Используем функцию get_prompt или build_prompt (смотря как ты её назвал выше)
# Если была build_prompt, оставь её. Если get_prompt — замени.
test_prompts = [get_prompt(item['original_fragment']) for item in test_items]

# Генерация
test_outputs = llm.generate(test_prompts, sampling_params)

for i, out in enumerate(test_outputs):
    neutral_text = out.outputs[0].text.strip()
    original_text = test_items[i]['original_fragment']
    
    # Оценка (используем уже загруженный embedder)
    emb1 = embedder.encode(original_text, convert_to_tensor=True)
    emb2 = embedder.encode(neutral_text, convert_to_tensor=True)
    score = torch.nn.functional.cosine_similarity(emb1.unsqueeze(0), emb2.unsqueeze(0)).item()
    
    print(f"\n[Пример #{i+1}] Score: {score:.4f}")
    print(f"ORIG: {original_text}...") # Увеличил до 200, чтобы было виднее
    print(f"NEUT: {neutral_text}...")
    print("-" * 50)

In [ ]:
# Если ты используешь ту же ячейку, просто введи 'y' в поле input под результатами теста.
# Или выполни этот блок, если он вынесен отдельно:

print(f"\n{'='*20} ЭТАП 2: ПОЛНАЯ ОБРАБОТКА ({len(dataset)} фрагментов) {'='*20}")

all_prompts = [build_prompt(item['original_fragment']) for item in dataset]
all_outputs = llm.generate(all_prompts, sampling_params)

neutral_texts = [out.outputs[0].text.strip() for out in all_outputs]
originals = [item['original_fragment'] for item in dataset]

print("\nВычисляю финальные метрики Topic Fidelity...")
# Пакетное вычисление эмбеддингов
emb_orig = embedder.encode(originals, batch_size=32, show_progress_bar=True, convert_to_tensor=True)
emb_neut = embedder.encode(neutral_texts, batch_size=32, show_progress_bar=True, convert_to_tensor=True)

scores = torch.nn.functional.cosine_similarity(emb_orig, emb_neut).tolist()

# Сборка финального JSON
for i, item in enumerate(dataset):
    item['neutral_proxy'] = neutral_texts[i]
    item['topic_fidelity'] = round(scores[i], 4)
    item['is_valid'] = scores[i] > 0.75

with open('/kaggle/working/dostoevsky_parallel_final.json', 'w', encoding='utf-8') as f:
    json.dump(dataset, f, ensure_ascii=False, indent=4)
    
print(f"\nГОТОВО! Файл сохранен: /kaggle/working/dostoevsky_parallel_final.json")

In [ ]:
import pandas as pd

df = pd.read_json('/kaggle/working/dostoevsky_parallel_final.json')

df_clean = df[df['is_valid'] == True].copy()

df_clean = df_clean[df_clean['original_fragment'] != df_clean['neutral_proxy']]

df_clean = df_clean[df_clean['neutral_proxy'].str.len() > 20]

# 4. Сохраняем "Золотой датасет" для обучения
FINAL_TRAIN_PATH = '/kaggle/working/dostoevsky_train_ready.jsonl'
df_clean[['original_fragment', 'neutral_proxy']].to_json(
    FINAL_TRAIN_PATH, orient='records', lines=True, force_ascii=False
)

print(f"Итоговая выборка для обучения LoRA: {len(df_clean)} из {len(df)}")

In [ ]:
# orient='records' создаст список объектов (как в твоем исходном JSON)
# indent=4 сделает файл читаемым (красивым)
df_clean.to_json('/kaggle/working/dostoevsky_parallel_final_LORA.json', 
                 orient='records', 
                 force_ascii=False, 
                 indent=4)

print(f"ГОТОВО! Файл сохранен: /kaggle/working/dostoevsky_parallel_final_LORA.json")

# Дообучение QWEN на перенос стиля Достоевского на существующий текст

In [ ]:
# Полная очистка
!pip uninstall -y unsloth unsloth_zoo trl peft accelerate bitsandbytes

# Установка стабильной релизной версии
!pip install --no-cache-dir "unsloth[colab-new]"
!pip install --no-cache-dir unsloth_zoo
!pip install --no-cache-dir "trl<0.13.0" peft accelerate bitsandbytes transformers

print("\n--- УСТАНОВКА ЗАВЕРШЕНА! ---")
print("Сделай Restart Session!")

In [ ]:
# Устанавливаем официальные релизные версии
!pip install --no-cache-dir "unsloth[colab-new]"
!pip install --no-cache-dir unsloth_zoo
!pip install --no-cache-dir "trl<0.13.0" peft accelerate bitsandbytes transformers

In [ ]:
!pip install --no-cache-dir --upgrade "transformers>=4.48.0"

In [1]:
import torch
import gc
# Очищаем всё, что застряло в памяти после ошибки
gc.collect()
torch.cuda.empty_cache()

In [2]:
import sys
import os

# --- ХАК ДЛЯ ОБХОДА БАГОВ UNSLOTH ---
try:
    # Создаем фиктивный модуль или правим существующий, чтобы избежать KeyError
    import unsloth_zoo.patching as patching
    if not hasattr(patching, "RL_REPLACEMENTS"):
        patching.RL_REPLACEMENTS = {}
    
    # Затыкаем дыры в словаре
    for key in ["sanitize_logprob", "align_logprobs_with_mask", "grpo_autotune_batch_and_chunks"]:
        if key not in patching.RL_REPLACEMENTS:
            patching.RL_REPLACEMENTS[key] = lambda x: x
    print("Патч словаря применен.")
except:
    # Если модуля еще нет, создаем заглушку в sys.modules
    from types import ModuleType
    m = ModuleType("unsloth_zoo.patching")
    m.RL_REPLACEMENTS = {
        "sanitize_logprob": lambda x: x,
        "align_logprobs_with_mask": lambda x: x,
        "grpo_autotune_batch_and_chunks": lambda x: x
    }
    sys.modules["unsloth_zoo.patching"] = m
    print("Создана системная заглушка для патчей.")

# ТЕПЕРЬ ИМПОРТИРУЕМ
from unsloth import FastLanguageModel
import torch

# --- ИНИЦИАЛИЗАЦИЯ QWEN 2.5 3B ---
max_seq_length = 2048
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-3B-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    load_in_4bit = True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 32, 
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 64,
    lora_dropout = 0,
    bias = "none",
)

print("Модель готова к обучению!")

Создана системная заглушка для патчей.
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


[unsloth_zoo.log|WARNING]Unsloth: Could not patch trl.trainer.ppov2_trainer: Direct module loading failed for UnslothPPOv2Trainer: duplicate argument 'kwargs' in function definition (UnslothPPOv2Trainer.py, line 341)


==((====))==  Unsloth 2026.2.1: Fast Qwen2 patching. Transformers: 5.2.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Unsloth 2026.2.1 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


Модель готова к обучению!


In [4]:
import json
import pandas as pd
from datasets import Dataset

# --- 1. ПОДГОТОВКА ДАННЫХ ---
INPUT_DATA = '/kaggle/input/datasets/kirillsilantev/lora12/dostoevsky_parallel_final_LORA.json'

with open(INPUT_DATA, 'r', encoding='utf-8') as f:
    data = json.load(f)

df = pd.DataFrame(data)

# Фильтрация
df_train = df[(df['is_valid'] == True) & (df['original_fragment'] != df['neutral_proxy'])].copy()

# СИСТЕМНЫЙ ПРОМПТ (Инструкция)
style_instruction = (
    "Ты — мастер литературной стилизации. Твоя задача — переписать предложенный текст, "
    "используя уникальный слог Ф.М. Достоевского. Сохраняй все факты, но наполни их "
    "характерным психологизмом, сложным синтаксисом и лексикой автора."
)

def formatting_prompts_func(examples):
    instructions = [style_instruction] * len(examples["neutral_proxy"])
    inputs       = examples["neutral_proxy"]
    outputs      = examples["original_fragment"]
    texts = []
    for instruction, input, output in zip(instructions, inputs, outputs):
        # Формируем структуру ChatML
        text = f"<|im_start|>system\n{instruction}<|im_end|>\n" \
               f"<|im_start|>user\n{input}<|im_end|>\n" \
               f"<|im_start|>assistant\n{output}<|im_end|>"
        texts.append(text)
    return { "text" : texts, }

# Превращаем в Dataset
dataset_hf = Dataset.from_pandas(df_train)

# Очистка колонок
column_names = dataset_hf.column_names
dataset_hf = dataset_hf.map(
    formatting_prompts_func, 
    batched = True,
    remove_columns = column_names 
)

# СОЗДАЕМ ЧИСТЫЙ ДАТАСЕТ (как мы договаривались для обхода ошибок)
final_train_dataset = Dataset.from_dict({"text": dataset_hf["text"]})

print(f"Готово к обучению на {len(final_train_dataset)} примерах.")
print(f"Колонки: {final_train_dataset.column_names}")

Map:   0%|          | 0/9832 [00:00<?, ? examples/s]

Готово к обучению на 9832 примерах.
Колонки: ['text']


In [6]:
def tokenize_fn(examples):
    return tokenizer(examples["text"], truncation=True, max_length=2048)

print("Токенизация... (это уберет все текстовые ошибки)")
tokenized_dataset = final_train_dataset.map(
    tokenize_fn,
    batched=True,
    remove_columns=["text"] # Удаляем текст, оставляем только числа
)

Токенизация... (это уберет все текстовые ошибки)


Map:   0%|          | 0/9832 [00:00<?, ? examples/s]

In [14]:
from trl import SFTTrainer, SFTConfig

# Настройка максимально аккуратного конфига
sft_config = SFTConfig(
    output_dir = "outputs_dostoevsky",
    max_seq_length = 2048,
    dataset_text_field = "text",
    num_train_epochs = 1,
    per_device_train_batch_size = 2, 
    gradient_accumulation_steps = 8, 
    gradient_checkpointing = True,   # Экономим память
    learning_rate = 2e-4,
    fp16 = True,
    logging_steps = 1,
    optim = "adamw_8bit",
    report_to = "none",
    # Важно: даем трейнеру самому решить, какие колонки удалять
    remove_unused_columns = True, 
)

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = final_train_dataset, # Твой датасет с колонкой 'text'
    args = sft_config,
    packing = True, # Этот режим в Unsloth работает стабильнее всего
)

# Принудительно отключаем кэш для обучения (это часто лечит OOM и ошибки лосса)
model.config.use_cache = False

print("\n" + "="*20 + " ЗАПУСК ОБУЧЕНИЯ (STABLE PACKING) " + "="*20)
trainer.train()

# Сохранение
model.save_pretrained_lora("dostoevsky_qwen_lora")
tokenizer.save_pretrained("dostoevsky_qwen_lora")

/tmp/unsloth_compiled_cache/UnslothSFTTrainer.py:756: UserWarning: You passed a `packing` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(


Generating train split: 0 examples [00:00, ? examples/s]

🦥 Unsloth: Packing enabled - training is >2x faster and uses less VRAM!

==================== ЗАПУСК ОБУЧЕНИЯ (STABLE PACKING) ====================


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,283 | Num Epochs = 1 | Total steps = 143
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 8 x 1) = 16
 "-____-"     Trainable parameters = 59,867,136 of 3,145,805,824 (1.90% trained)


Step,Training Loss
1,1.856544
2,1.829358
3,1.807345
4,1.813008
5,1.712727
6,1.671463
7,1.655287
8,1.698028
9,1.638829
10,1.615964


AttributeError: 'Qwen2ForCausalLM' object has no attribute 'save_pretrained_lora'

In [15]:
# Стандартный способ сохранения для PEFT моделей
model.save_pretrained("dostoevsky_qwen_lora")
tokenizer.save_pretrained("dostoevsky_qwen_lora")

print("Всё! Модель успешно сохранена в папку dostoevsky_qwen_lora")

Всё! Модель успешно сохранена в папку dostoevsky_qwen_lora


In [16]:
from unsloth import FastLanguageModel

# Переключаем в режим предсказания
FastLanguageModel.for_inference(model)

prompt = f"<|im_start|>system\n{style_instruction}<|im_end|>\n<|im_start|>user\nСегодня я зашел в магазин, купил хлеба и почувствовал какую-то странную тоску, глядя на прохожих.<|im_end|>\n<|im_start|>assistant\n"

inputs = tokenizer([prompt], return_tensors = "pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens = 150, temperature = 0.7)
print(tokenizer.decode(outputs[0], skip_special_tokens = True))

--- Logging error ---
Traceback (most recent call last):
  File "/usr/lib/python3.12/logging/__init__.py", line 1160, in emit
    msg = self.format(record)
          ^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/logging/__init__.py", line 999, in format
    return fmt.format(record)
           ^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/logging/__init__.py", line 703, in format
    record.message = record.getMessage()
                     ^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/logging/__init__.py", line 392, in getMessage
    msg = msg % self.args
          ~~~~^~~~~~~~~~~
TypeError: not all arguments converted during string formatting
Call stack:
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py", line 37, in <module>
    ColabKernelApp.launch_instance()
  File "/usr/local/lib/python3.12/dist-packages/traitlets/config/application.py", line 992, 

system
Ты — мастер литературной стилизации. Твоя задача — переписать предложенный текст, используя уникальный слог Ф.М. Достоевского. Сохраняй все факты, но наполни их характерным психологизмом, сложным синтаксисом и лексикой автора.
user
Сегодня я зашел в магазин, купил хлеба и почувствовал какую-то странную тоску, глядя на прохожих.
assistant
Сегодняшний день… Я вошел в магазин, взял хлеба, а глядя на прохожих, стал как бы вдруг что-то терять.


In [18]:
test_cases_deep = [
    "Я стою у порога своего кабака и смотрю на свои старые сапоги, через которые уже видна голая земля. Денег нет совсем, в кармане лишь пустота, а дома ждут голодные дети. Хозяйка вчера кричала на всю лестницу, обещала выкинуть мои пожитки на мостовую. Я чувствую себя последним человеком, ветошкой, об которую каждый волен вытереть ноги.",
    
    "Старик Мармеладов сидел в углу, уронив голову на грязный стол, и тихо всхлипывал. Его шинель превратилась в нечто невообразимое, лохмотья едва прикрывали худые плечи. Вокруг смеялись пьяные мужики, тыкали в него пальцами, а он только смиренно крестился. В его глазах застыла такая бездна отчаяния, что мне стало страшно за свою собственную мелкую душу.",
    
    "Чиновник шел по мосту, прижимая к груди тонкую папку с бумагами, и дрожал от пронизывающего ветра. Он думал о том, что завтра ему нечего будет надеть на службу, а пуговицы на мундире едва держатся. Проходящие мимо господа в богатых экипажах обдавали его грязью из луж, но он даже не поднимал глаз. Вся его жизнь казалась ему случайной ошибкой, коротким сном в холодном петербургском тумане."
]

print("="*40 + " ЛИТЕРАТУРНЫЕ ЧТЕНИЯ " + "="*40)

for i, user_text in enumerate(test_cases_deep, 1):
    # Увеличиваем max_new_tokens, чтобы модель не обрывала мысль
    prompt = f"<|im_start|>system\n{style_instruction}<|im_end|>\n<|im_start|>user\n{user_text}<|im_end|>\n<|im_start|>assistant\n"
    
    inputs = tokenizer([prompt], return_tensors = "pt").to("cuda")
    outputs = model.generate(
        **inputs, 
        max_new_tokens = 450, 
        temperature = 0.85, 
        top_p = 0.92, 
        repetition_penalty = 1.2,
        do_sample = True
    )
    
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens = True).split("assistant\n")[-1]
    
    print(f"\nВАРИАНТ №{i} (ИСХОДНИК):\n{user_text}")
    print(f"\nВАРИАНТ №{i} (СТИЛИЗАЦИЯ):\n{generated_text.strip()}")
    print("-" * 80)

======================================== ЛИТЕРАТУРНЫЕ ЧТЕНИЯ ========================================

ВАРИАНТ №1 (ИСХОДНИК):
Я стою у порога своего кабака и смотрю на свои старые сапоги, через которые уже видна голая земля. Денег нет совсем, в кармане лишь пустота, а дома ждут голодные дети. Хозяйка вчера кричала на всю лестницу, обещала выкинуть мои пожитки на мостовую. Я чувствую себя последним человеком, ветошкой, об которую каждый волен вытереть ноги.

ВАРИАНТ №1 (СТИЛИЗАЦИЯ):
Один я стоит при своем собственном кабаке и осматриваю свои старый-старенький кожанчик, чрез который теперь видно уже белеющую зелень подножья дороги. В кармане тоже ничего не оказалось – только пустота, да еще домой душатся голодными дети… Хозяйка давеча кричит со всей своей горлышечкою по всемамерному коридору, чтобы они мне выгонят всякие «пожитки». Сам же я себе за последние месяцы так мало набирал денег, что даже сейчас, когда надо бы было отворить мой кошелек, чтоб навязать хозяеваши тем самым каких-ни